<a href="https://colab.research.google.com/github/EmperorBlackMD/BME-6720/blob/main/BME6720_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Week 1: MSD Spleen Dataset Initial Check
import os
import json
import pandas as pd
import tarfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Setting dataset path
tar_path = Path('/content/drive/MyDrive/BME6720_Project/Task09_Spleen.tar')
DATASET_DIR = Path('/content/drive/MyDrive/BME6720_Project/')

with tarfile.open(tar_path, 'r') as tar:
    tar.extractall(path=DATASET_DIR)
print('Extraction complete')

for root, dirs, files in os.walk(DATASET_DIR):
    print(root)


/tmp/ipykernel_2292/1469869667.py:6: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=DATASET_DIR)


Extraction complete
/content/drive/MyDrive/BME6720_Project
/content/drive/MyDrive/BME6720_Project/Task09_Spleen
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/labelsTr
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/imagesTr
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/imagesTs


In [6]:
# Finalizing dataset path

DATASET_DIR1 = Path('/content/drive/MyDrive/BME6720_Project/Task09_Spleen')

imagesTr_dir = DATASET_DIR1 / 'imagesTr'
labelsTr_dir = DATASET_DIR1 / 'labelsTr'
imagesTs_dir = DATASET_DIR1 / 'imagesTs'
dataset_json_path = DATASET_DIR1 / 'dataset.json'

expected_paths = {
    'Dataset root': DATASET_DIR1,
    'Training images folder': imagesTr_dir,
    'Training labels folder': labelsTr_dir,
    'Test images folder': imagesTs_dir,
    'Dataset JSON': dataset_json_path
}

print('Dataset structure check')
for name, path in expected_paths.items():
    print(f'{name}: {'FOUND' if path.exists() else 'NOT FOUND'} --> {path}')

Dataset structure check
Dataset root: FOUND --> /content/drive/MyDrive/BME6720_Project/Task09_Spleen
Training images folder: FOUND --> /content/drive/MyDrive/BME6720_Project/Task09_Spleen/imagesTr
Training labels folder: FOUND --> /content/drive/MyDrive/BME6720_Project/Task09_Spleen/labelsTr
Test images folder: FOUND --> /content/drive/MyDrive/BME6720_Project/Task09_Spleen/imagesTs
Dataset JSON: FOUND --> /content/drive/MyDrive/BME6720_Project/Task09_Spleen/dataset.json


In [7]:
# List image and label files
train_images = sorted(list(imagesTr_dir.glob('*.nii.gz')))
train_labels = sorted(list(labelsTr_dir.glob('*.nii.gz')))
test_images = sorted(list(imagesTs_dir.glob('*.nii.gz')))

print(f'Number of Training images: {len(train_images)}')
print(f'Number of Training labels: {len(train_labels)}')
print(f'Number of Test images: {len(test_images)}')

print('\nFirst 5 Training images:')
for image in train_images[:5]:
    print(image)

print('\nFirst 5 Training labels:')
for label in train_labels[:5]:
    print(label)


Number of Training images: 82
Number of Training labels: 82
Number of Test images: 40

First 5 Training images:
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/imagesTr/._spleen_10.nii.gz
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/imagesTr/._spleen_12.nii.gz
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/imagesTr/._spleen_13.nii.gz
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/imagesTr/._spleen_14.nii.gz
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/imagesTr/._spleen_16.nii.gz

First 5 Training labels:
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/labelsTr/._spleen_10.nii.gz
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/labelsTr/._spleen_12.nii.gz
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/labelsTr/._spleen_13.nii.gz
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/labelsTr/._spleen_14.nii.gz
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/labelsTr/._spleen_16.nii.gz


In [8]:
# Read dataset.json metadata
with open(dataset_json_path, 'r') as f:
    dataset_json = json.load(f)

print('Dataset metadata:')
print(json.dumps(dataset_json, indent=4))


Dataset metadata:
{
    "name": "Spleen",
    "description": "Spleen Segmentation",
    "reference": "Memorial Sloan Kettering Cancer Center",
    "licence": "CC-BY-SA 4.0",
    "release": "1.0 06/08/2018",
    "tensorImageSize": "3D",
    "modality": {
        "0": "CT"
    },
    "labels": {
        "0": "background",
        "1": "spleen"
    },
    "numTraining": 41,
    "numTest": 20,
    "training": [
        {
            "image": "./imagesTr/spleen_19.nii.gz",
            "label": "./labelsTr/spleen_19.nii.gz"
        },
        {
            "image": "./imagesTr/spleen_31.nii.gz",
            "label": "./labelsTr/spleen_31.nii.gz"
        },
        {
            "image": "./imagesTr/spleen_52.nii.gz",
            "label": "./labelsTr/spleen_52.nii.gz"
        },
        {
            "image": "./imagesTr/spleen_40.nii.gz",
            "label": "./labelsTr/spleen_40.nii.gz"
        },
        {
            "image": "./imagesTr/spleen_3.nii.gz",
            "label": "./labelsTr

In [11]:
# Creating case-level manifest

def get_case_id(filename):
    """
    Extracts case ID from MSD-style filenames.
    Example: spleen_10.nii.gz -> spleen_10
    """
    return filename.replace('.nii.gz', '')

image_case_ids = [get_case_id(image.name) for image in train_images]
label_case_ids = [get_case_id(label.name) for label in train_labels]
manifest = []

for image_file in train_images:
    case_id = get_case_id(image_file.name)
    expected_label = labelsTr_dir / image_file.name

    manifest.append({
        'case_id': case_id,
        'image_file': str(image_file),
        'label_file': str(expected_label),
        'label_exists': expected_label.exists()
    })
manifest_df = pd.DataFrame(manifest)
manifest_df.head()

,case_id,image_file,label_file,label_exists
0,._spleen_10,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True
1,._spleen_12,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True
2,._spleen_13,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True
3,._spleen_14,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True
4,._spleen_16,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True


In [12]:
from IPython.core.interactiveshell import dis
# Check for missing labels or unmatched files
missing_labels = manifest_df[manifest_df['label_exists'] == False]

print(f'Total image-label pairs expected: {len(manifest_df)}')
print(f'Total missing label files: {len(missing_labels)}')

if len(missing_labels) > 0:
    display(missing_labels)
else:
    print('All training images have corresponding labels')


Total image-label pairs expected: 82
Total missing label files: 0
All training images have corresponding labels


In [13]:
# Saving manifest for future weekly updates
output_manifest_path = DATASET_DIR1 / 'week1dataset_manifest.csv'
manifest_df.to_csv(output_manifest_path, index=False)
print(f'Manifest saved to {output_manifest_path}')

Manifest saved to /content/drive/MyDrive/BME6720_Project/Task09_Spleen/week1dataset_manifest.csv
